# Lesson 06: Answers and Verify (solution)

**Goal:** build the LLM context, get a structured answer and watch Verify reject an invented quote.

**Before you start**
- Theory: [lesson 06](../../../docs/lessons/06_answer_and_verify.md)
- Python concepts: Pydantic models as `response_format`, string handling, `await`
- Files used: `fixtures/tiny.pdf`, the fake LLM. Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
from pathlib import Path

from docrag.answer.context import load_prompt, render_context
from docrag.answer.schema import Answer, Evidence
from docrag.answer.verify import verify
from docrag.ingest.pipeline import ingest_pdf
from docrag.llm.fake import FakeProvider
from docrag.llm.proxy import LLMProxy
from docrag.llm.types import Message
from docrag.shared.schema import Hit

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
chunks = ingest_pdf(ROOT / "fixtures" / "tiny.pdf", "tiny")
# Pretend retrieval returned the table first (score/source only matter for debugging).
hits = [
    Hit(chunk=c, score=1.0, source="rerank")
    for c in sorted(chunks, key=lambda c: c.type != "table")
]
proxy = LLMProxy({"fake": FakeProvider()})
question = "What was the revenue in Europe in 2024?"

## Step 1: Render the context

The LLM sees each chunk as `[chunk id] (page, type, section) text`. The ID in brackets is the **citation handle**: the model must cite it, and Verify later checks it. `render_context(hits)` builds that block.

In [ ]:
context = render_context(hits)

In [ ]:
assert "[tiny:p2:c1]" in context, "Every chunk starts with its ID in brackets"
assert context.index("[tiny:p2:c1]") < context.index("[tiny:p1:c0]"), (
    "Order = retrieval order"
)
print(context[:400])
print("OK")

## Step 2: A structured answer

`proxy.structured(route, messages, Answer)` asks for JSON matching the Pydantic model `Answer` (`answer` + a list of `evidence` items with `id` and `quote`) and returns `(parsed, completion)`. Invalid JSON gets exactly one retry with the validation error. Static instructions go first (providers cache identical prompt prefixes).

In [ ]:
messages = [
    Message(role="system", content=f"{load_prompt('answer_v1')}\n{context}"),
    Message(role="user", content=question),
]
answer, completion = await proxy.structured("fake:m", messages, Answer)

In [ ]:
assert isinstance(answer, Answer), (
    "Hint: await proxy.structured('fake:m', messages, Answer)"
)
assert answer.evidence, "The fake LLM cites the first passage with a real quote"
result = verify(question, answer, chunks)
print(answer.answer)
print("Verify:", result.status, [(c.id, c.verified) for c in result.citations])
assert result.status == "supported"
print("OK")

## Step 3: An invented quote

Verify checks two things per citation: the ID was in the context, and the quote appears **verbatim** (after normalizing case, quotes and whitespace) in that chunk. Build an `Answer` that cites the table with a quote that isn't there, and verify it.

In [ ]:
invented = Answer(
    answer="Revenue tripled [tiny:p2:c1]",
    evidence=[Evidence(id="tiny:p2:c1", quote="revenue tripled in Europe")],
)
bad = verify(question, invented, chunks) if invented else None

In [ ]:
assert bad is not None, "Build the Answer first"
assert bad.status == "unsupported", (
    f"No verified citation left -> unsupported, got {bad.status}"
)
assert bad.citations[0].reason == "quote not found in chunk"
assert "[tiny:p2:c1]" not in bad.answer, (
    "The inline marker of a rejected citation is removed"
)
print(bad.status, "|", bad.citations[0].reason, "|", repr(bad.answer))
print("OK")

## Observation question

When is an answer "supported" even though it is wrong?

<details>
<summary>Answer</summary>

Verify checks that a quote is *real*, not that it *supports* the claim. Try it:

```python
wrong = Answer(answer="Europe had 24.9 million in 2024 [tiny:p2:c1]",
               evidence=[Evidence(id="tiny:p2:c1", quote="|Asia|18.2|24.9|")])
verify(question, wrong, chunks).status   # 'supported'
```

The quote exists in the table, so the citation passes, but it's the Asia row. Checking that a quote entails the claim needs a model (an NLI model or LettuceDetect), which is the planned second Verify stage. The LLM judge in the eval (`faithful`) catches this kind of error for the benchmark.
</details>

## The real code

- Context rendering: [`src/docrag/answer/context.py`](../../../src/docrag/answer/context.py)
- AnswerService: [`src/docrag/answer/service.py`](../../../src/docrag/answer/service.py)
- Verify: [`src/docrag/answer/verify.py`](../../../src/docrag/answer/verify.py)
- Prompt: [`src/docrag/prompts/answer_v1.txt`](../../../src/docrag/prompts/answer_v1.txt)
- Lesson: [`docs/lessons/06_answer_and_verify.md`](../../../docs/lessons/06_answer_and_verify.md)